In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Drag the two sliders. On the left, the gray curve is the true tone, the
red dots are the samples taken at $f_s$, and the gold curve is the alias
that runs through the very same dots. On the right, the gold line maps
every true frequency to the one you actually hear. The audio card plays
the current setting.

In [ ]:
# hide
# autorun
F0, FS0 = 300.0, 1400.0             # starting parameters

T_WIN = 0.006                       # six milliseconds on screen
t = np.linspace(0.0, T_WIN, 1200)
SR = 44100                          # playback rate
T_PLAY = np.arange(int(1.2 * SR)) / SR
FGRID = np.linspace(0.0, 1250.0, 700)

def alias(f, fs):
    # the chapter's formula, plus the sign the reconstruction comes back with
    m = np.mod(f, fs)
    return min(m, fs - m), (1.0 if m <= fs / 2 else -1.0)

def fold(fs, grid=FGRID):
    m = np.mod(grid, fs)
    return np.minimum(m, fs - m)

def samples(f, fs):
    n = np.arange(int(np.ceil(T_WIN * fs)) + 1)
    return n / fs, np.sin(2 * np.pi * f * n / fs)

def figure():
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.13)
    fa, sign = alias(F0, FS0)
    xs, ys = samples(F0, FS0)
    fig.add_scatter(x=t * 1000, y=np.sin(2 * np.pi * F0 * t), mode="lines",
                    line=dict(color=STEEL, width=1.6), row=1, col=1)
    fig.add_scatter(x=t * 1000, y=sign * np.sin(2 * np.pi * fa * t),
                    mode="lines", line=dict(color=GOLD, width=2.2),
                    row=1, col=1)
    fig.add_scatter(x=xs * 1000, y=ys, mode="markers",
                    marker=dict(color=RED, size=7), row=1, col=1)
    fig.add_scatter(x=[0, 1250], y=[0, 1250], mode="lines",
                    line=dict(color=STEEL, width=1.2, dash="dot"),
                    row=1, col=2)
    fig.add_scatter(x=FGRID, y=fold(FS0), mode="lines",
                    line=dict(color=GOLD, width=2.2), row=1, col=2)
    fig.add_scatter(x=[F0], y=[fa], mode="markers",
                    marker=dict(color=RED, size=12,
                                line=dict(color="white", width=2)),
                    row=1, col=2)
    fig.update_xaxes(range=[0, T_WIN * 1000], title_text="Time (ms)",
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=[-1.15, 1.15], title_text="Amplitude",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, 1250], title_text="True frequency (Hz)",
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[0, 1250], title_text="Heard frequency (Hz)",
                     fixedrange=True, row=1, col=2)
    return fig

def controls(fig):
    f = widgets.FloatSlider(description="Tone $f$ (Hz)", min=50, max=1200,
                            value=F0, step=10)
    fs = widgets.FloatSlider(description="Sample rate $f_s$ (Hz)", min=400,
                             max=2400, value=FS0, step=50)
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(f, fs, t=t, alias=alias, fold=fold, samples=samples,
               readout=readout):
        fa, sign = alias(f, fs)
        xs, ys = samples(f, fs)
        with fig.batch_update():
            fig.data[0].y = np.sin(2 * np.pi * f * t)
            fig.data[1].y = sign * np.sin(2 * np.pi * fa * t)
            fig.data[2].x, fig.data[2].y = xs * 1000, ys
            fig.data[4].y = fold(fs)
            fig.data[5].x, fig.data[5].y = [f], [fa]
        note = ("no aliasing" if abs(fa - f) < 1e-9
                else f"heard as <i>f</i><sub>alias</sub> = {fa:.0f} Hz")
        readout.value = (f"<span style='font-size:0.9em'><i>f</i><sub>s</sub>/2 = "
                         f"{fs / 2:.0f} Hz &nbsp;·&nbsp; {note}</span>")

    widgets.interactive_output(update, {"f": f, "fs": fs})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def render(T_PLAY=T_PLAY, SR=SR, alias=alias):
        fa, sign = alias(f.value, fs.value)
        # reconstructing those samples gives back exactly one sinusoid, the
        # alias, so we synthesize it directly at the playback rate
        x = sign * 0.125 * np.sin(2 * np.pi * fa * T_PLAY)   # about -18 dBFS
        x[:441] *= np.linspace(0, 1, 441)
        x[-441:] *= np.linspace(1, 0, 441)
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)


    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (f, fs):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([f, fs, readout, out, gate])

icm_plotly.show(figure, controls)